In [ ]:
# ============================================================
# MULTI-SEED ANALYSIS (best fold) + PER-CLASS RESULTS
# ============================================================
import random, copy

FOLD_ID = 0
SEEDS   = [42, 123, 2024, 7, 999]
MS_DIR  = os.path.join(OUT_DIR, f"multiseed_fold{FOLD_ID}")
os.makedirs(MS_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

ms_overall   = []      # per-seed overall metrics
ms_perclass  = []      # per-seed per-class metrics
ms_cms       = {}
ms_hists     = {}
ms_probs     = {}

_orig_out_dir = OUT_DIR

for seed in SEEDS:
    print(f"\n{'='*70}\n Fold {FOLD_ID} | Seed {seed}\n{'='*70}")
    set_seed(seed)

    
    seed_dir = os.path.join(MS_DIR, f"seed{seed}")
    os.makedirs(seed_dir, exist_ok=True)
    OUT_DIR = seed_dir
    try:
        model, hist = train_one_fold(FOLD_ID)
    finally:
        OUT_DIR = _orig_out_dir

    ms_hists[seed] = hist
    hist.to_csv(os.path.join(seed_dir, "history.csv"), index=False)

    # ---- Test evaluation ----
    te_loader = make_loader(folds[FOLD_ID]["test"], eval_tf, False)
    probs, labels, preds = evaluate(model, te_loader)
    ms_probs[seed] = (probs, labels)
    ms_cms[seed]   = confusion_matrix(labels, preds)

    # ---- Per-class metrics ----
    pc = per_class_metrics(labels, preds, CLASSES)
    pc.insert(0, "Seed", seed)
    pc.to_csv(os.path.join(seed_dir, "per_class_metrics.csv"), index=False)
    ms_perclass.append(pc)

    acc = accuracy_score(labels, preds)
    mf1 = f1_score(labels, preds, average="macro")
    wf1 = f1_score(labels, preds, average="weighted")
    mp  = precision_score(labels, preds, average="macro")
    mr  = recall_score(labels, preds, average="macro")
    ms_overall.append({"seed": seed, "accuracy": acc, "macro_precision": mp,
                       "macro_recall": mr, "macro_f1": mf1, "weighted_f1": wf1})

    print(f"\n--- Seed {seed} Per-Class ---")
    print(pc.drop(columns="Seed").to_string(index=False))
    print(f"Seed {seed} → Acc {acc:.4f} | Macro-F1 {mf1:.4f} | Weighted-F1 {wf1:.4f}")

    del model
    torch.cuda.empty_cache()

# ============================================================
# AGGREGATE RESULTS
# ============================================================
ov_df = pd.DataFrame(ms_overall)
pc_df = pd.concat(ms_perclass, ignore_index=True)

print("\n" + "="*70)
print(f"MULTI-SEED SUMMARY (Fold {FOLD_ID}, {len(SEEDS)} seeds)")
print("="*70)
print(ov_df.round(4).to_string(index=False))
print("-"*70)
for c in ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]:
    print(f"{c:16s}: {ov_df[c].mean():.4f} ± {ov_df[c].std():.4f}")

# ---- Per-class: mean ± std across seeds ----
metric_cols = ["Accuracy (%)", "TPR (%)", "FNR (%)", "FPR (%)",
               "TNR (%)", "Precision (%)", "F1 Score (%)", "Error Rate (%)"]
g = pc_df.groupby("Class", sort=False)[metric_cols]
mean_df, std_df = g.mean(), g.std()

pc_summary = pd.DataFrame(index=mean_df.index)
for m in metric_cols:
    pc_summary[m] = mean_df[m].round(2).astype(str) + " ± " + std_df[m].round(2).astype(str)
pc_summary = pc_summary.reset_index()

print("\n" + "="*70)
print("PER-CLASS RESULTS (mean ± std over seeds)")
print("="*70)
print(pc_summary.to_string(index=False))

# ---- Save ----
ov_df.to_csv(os.path.join(MS_DIR, "overall_per_seed.csv"), index=False)
pc_df.to_csv(os.path.join(MS_DIR, "per_class_all_seeds.csv"), index=False)
mean_df.round(2).reset_index().to_csv(os.path.join(MS_DIR, "per_class_mean.csv"), index=False)
std_df.round(2).reset_index().to_csv(os.path.join(MS_DIR, "per_class_std.csv"), index=False)
pc_summary.to_csv(os.path.join(MS_DIR, "per_class_mean_pm_std.csv"), index=False)

# ============================================================
# PLOTS
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# 1. Val macro-F1 curves per seed
for s in SEEDS:
    axes[0, 0].plot(ms_hists[s]["epoch"], ms_hists[s]["val_f1"], lw=2, label=f"Seed {s}")
axes[0, 0].set(title="Validation Macro-F1 (per seed)", xlabel="Epoch", ylabel="Macro-F1")
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# 2. Train loss per seed
for s in SEEDS:
    axes[0, 1].plot(ms_hists[s]["epoch"], ms_hists[s]["train_loss"], lw=2, label=f"Seed {s}")
axes[0, 1].set(title="Training Loss (per seed)", xlabel="Epoch", ylabel="Loss")
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# 3. Test overall metrics per seed
x = np.arange(len(SEEDS)); w = 0.25
axes[0, 2].bar(x - w, ov_df["accuracy"]*100,    w, label="Accuracy")
axes[0, 2].bar(x,     ov_df["macro_f1"]*100,    w, label="Macro-F1")
axes[0, 2].bar(x + w, ov_df["weighted_f1"]*100, w, label="Weighted-F1")
axes[0, 2].set_xticks(x); axes[0, 2].set_xticklabels([f"S{s}" for s in SEEDS])
axes[0, 2].set(title="Test Metrics per Seed", ylabel="%"); axes[0, 2].legend()
axes[0, 2].grid(axis="y", alpha=0.3)

# 4. Per-class F1 (mean ± std)
axes[1, 0].bar(mean_df.index, mean_df["F1 Score (%)"], yerr=std_df["F1 Score (%)"],
               capsize=6, color=plt.cm.Set2.colors[:len(CLASSES)], edgecolor="black")
axes[1, 0].set(title="Per-Class F1 (mean ± std)", ylabel="F1 (%)")
axes[1, 0].grid(axis="y", alpha=0.3)

# 5. Per-class TPR (mean ± std)
axes[1, 1].bar(mean_df.index, mean_df["TPR (%)"], yerr=std_df["TPR (%)"],
               capsize=6, color=plt.cm.Set2.colors[:len(CLASSES)], edgecolor="black")
axes[1, 1].set(title="Per-Class TPR/Recall (mean ± std)", ylabel="TPR (%)")
axes[1, 1].grid(axis="y", alpha=0.3)

# 6. Mean confusion matrix (row-normalized)
cm_mean = np.mean([ms_cms[s] / ms_cms[s].sum(1, keepdims=True) for s in SEEDS], axis=0)
sns.heatmap(cm_mean, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1, 2])
axes[1, 2].set(title="Mean Normalized Confusion Matrix", xlabel="Predicted", ylabel="True")

plt.suptitle(f"DentAttn-MSFNet | Fold {FOLD_ID} | Multi-Seed Analysis ({len(SEEDS)} seeds)",
             fontsize=16, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(MS_DIR, "multiseed_analysis.png"), dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ Done! Saved in: {MS_DIR}")